In [1]:
import json
import numpy as np
import pandas as pd
from io import StringIO
import textwrap
from model_inference.gpt import *
from utils.table_utils import *

# Table parsing test

In [2]:
path = '../data/livesum/test.json'

In [3]:
df = pd.read_json(path)

In [4]:
print(df.head())

                                                text  \
0  And we're off for the first half. Player27(Awa...   
1  The game is underway with the start of the fir...   
2  The game is underway with the start of the fir...   
3  And we're off for the first half. Player26(Awa...   
4  The game is underway with the start of the fir...   

                                               table        id  
0  Team,Goals,Shots,Fouls,Yellow Cards,Red Cards,...  25513332  
1  Team,Goals,Shots,Fouls,Yellow Cards,Red Cards,...  25513360  
2  Team,Goals,Shots,Fouls,Yellow Cards,Red Cards,...  25600389  
3  Team,Goals,Shots,Fouls,Yellow Cards,Red Cards,...  25617902  
4  Team,Goals,Shots,Fouls,Yellow Cards,Red Cards,...  25892175  


In [5]:
print(df.columns)

Index(['text', 'table', 'id'], dtype='object')


In [6]:
idx = 1

In [7]:
print(df['table'][idx])

Team,Goals,Shots,Fouls,Yellow Cards,Red Cards,Corner Kicks,Free Kicks,Offsides<NEWLINE>Away Team,0,8,11,2,0,2,6,3<NEWLINE>Home Team,2,28,6,1,0,12,11,3


In [8]:
table_string = df['table'][idx]
table_string = table_string.replace('<NEWLINE>', '\n')

In [9]:
table_string_io = StringIO(table_string)

In [10]:
df_table = pd.read_csv(table_string_io)

In [11]:
print(df_table.to_string(index=False))

     Team  Goals  Shots  Fouls  Yellow Cards  Red Cards  Corner Kicks  Free Kicks  Offsides
Away Team      0      8     11             2          0             2           6         3
Home Team      2     28      6             1          0            12          11         3


# Prompting test

In [12]:
df = pd.read_json('../data/livesum/test.json')
%clear
print(textwrap.fill(df['text'][idx], width=100))

The game is underway with the start of the first half. Player8(Home Team) commits a foul, giving
Player26(Away Team) a free kick in the attacking half, . Offside called against the Home Team as
Player10(Home Team) attempts a through ball, but Player7(Home Team) is caught offside. Player28(Away
Team) commits a foul. Player9(Home Team) earns a free kick on the right side of the field.
Player8(Home Team) attempts a through ball, but Player7(Home Team) is offside. Player28(Away Team)
earns a free kick in the opponent's half. Player11(Home Team) committed a foul. Player29(Away Team)
from the Away Team attempts a through ball, but Player24(Away Team) is flagged for offside.
Player28(Away Team) attempts a through ball, but Player27(Away Team) is offside for the Away Team.
Player7(Home Team) misses a close shot from the right side of the six yard box, assisted by
Player10(Home Team) with a cross. Player22(Away Team) commits a foul. Player4(Home Team) earns a
free kick in the opponent's half. P

In [13]:
text = df['text'][idx]
atomic_out = ask_chatgpt(text=text,prompt_path="prompts/livesum_atomic.txt")
print(atomic_out)

The game is underway with the start of the first half. Player8 from the Home Team commits a foul. Player26 from the Away Team is awarded a free kick in the attacking half. An offside is called against the Home Team. Player10 from the Home Team attempts a through ball. Player7 from the Home Team is caught offside. Player28 from the Away Team commits a foul. Player9 from the Home Team earns a free kick on the right side of the field. Player8 from the Home Team attempts a through ball. Player7 from the Home Team is offside. Player28 from the Away Team earns a free kick in the opponent's half. Player11 from the Home Team commits a foul. Player29 from the Away Team attempts a through ball. Player24 from the Away Team is flagged for offside. Player28 from the Away Team attempts a through ball. Player27 from the Away Team is offside. Player7 from the Home Team misses a close shot from the right side of the six-yard box. Player10 from the Home Team assists Player7 with a cross. Player22 from t

In [14]:
with open('./model_outputs/gpt_livesum_test/atomic_each.txt', 'w') as f:
    f.write(atomic_out)

In [15]:
with open('./model_outputs/gpt_livesum_test/atomic_each.txt', 'r') as f:
    atomic_text = f.read()
header_out = ask_chatgpt(text=atomic_text,prompt_path="prompts/livesum_header.txt")
print(header_out)

{
  "row_headers": ["Event Number", "Player", "Team", "Action", "Location", "Result", "Time"],
  "column_headers": ["Event Type", "Foul", "Free Kick", "Offside", "Shot", "Assist", "Corner Kick", "Goal", "Yellow Card", "Injury Delay"]
}


In [16]:
with open('./model_outputs/gpt_livesum_test/header_each.txt', 'w') as f:
    f.write(header_out)

In [17]:
with open('./model_outputs/gpt_livesum_test/header_each.txt', 'r') as f:
    header_text = f.read()
with open('./model_outputs/gpt_livesum_test/atomic_each.txt', 'r') as f:
    atomic_text = f.read()
input_text = header_text + '\n' + atomic_text
output_table = ask_chatgpt(text=input_text,prompt_path="prompts/livesum_table.txt")
print(output_table)

|  | Event Type | Foul | Free Kick | Offside | Shot | Assist | Corner Kick | Goal | Yellow Card | Injury Delay |
| Event Number | 1 | 1 | 1 | 3 | 1 | 1 | 2 | 0 | 1 | 1 |
| Player | Player8 | Player28 | Player9 | Player7 | Player2 | Player10 | Player4 | Player15 | Player5 | Player25 |
| Team | Home Team | Away Team | Home Team | Home Team | Home Team | Home Team | Home Team | Home Team | Home Team | Away Team |
| Action | Commits | Awarded | Earns | Caught | Misses | Assists | Wins | Scores | Receives | Causing |
| Location | Not found | Attacking half | Right side | Not found | Right side of the box | Not found | Not found | Right side of the six-yard box | Not found | Not found |
| Result | Not found | Not found | Not found | Not found | Not found | Not found | Not found | Not found | Not found | Not found |
| Time | Not found | Not found | Not found | Not found | Not found | Not found | Not found | Not found | Not found | Not found |
<NEWLINE>
|  | Event Type | Foul | Free Kick | Off

In [18]:
convert_to_df(output_table)

,,Event Type,Foul,Free Kick,Offside,Shot,Assist,Corner Kick,Goal,Yellow Card,Injury Delay
0,Event Number,1,1,1,3,1,1,2,0,1,1
1,Player,Player8,Player28,Player9,Player7,Player2,Player10,Player4,Player15,Player5,Player25
2,Team,Home Team,Away Team,Home Team,Home Team,Home Team,Home Team,Home Team,Home Team,Home Team,Away Team
3,Action,Commits,Awarded,Earns,Caught,Misses,Assists,Wins,Scores,Receives,Causing
4,Location,Not found,Attacking half,Right side,Not found,Right side of the box,Not found,Not found,Right side of the six-yard box,Not found,Not found
...,...,...,...,...,...,...,...,...,...,...,...
83,Team,Away Team,Away Team,Away Team,Home Team,Home Team,Home Team,Home Team,Home Team,Away Team,Home Team
84,Action,Commits,Earns,Earns,Assists,Assists,Assists,Wins,Scores,Commits,Holding up
85,Location,Not found,Left wing,Own half,Not found,Not found,Not found,Not found,Right side of the box,Not found,Not found
86,Result,Not found,Not found,Not found,Not found,Not found,Not found,Not found,Not found,Not found,Not found


In [19]:
print(df_table.to_string(index=False))

     Team  Goals  Shots  Fouls  Yellow Cards  Red Cards  Corner Kicks  Free Kicks  Offsides
Away Team      0      8     11             2          0             2           6         3
Home Team      2     28      6             1          0            12          11         3
